# Thai Election OCR Pipeline (PaddleOCR)
## Super AI Engineer Season 6 - OCR Competition 2569

Extract structured voting data from scanned Thai election result documents.

**Pipeline:**
1. Download data from Kaggle
2. Group multi-page documents
3. OCR with PaddleOCR (Thai) + coordinate-based table extraction
4. Fuzzy match party names to template
5. Generate submission CSV

**Runtime: GPU (T4)** - ไปที่ Runtime > Change runtime type > GPU

In [ ]:
# === Cell 1: Install Dependencies ===
!pip install -q paddlepaddle-gpu paddleocr rapidfuzz tqdm 2>/dev/null || \
  pip install -q paddlepaddle paddleocr rapidfuzz tqdm
print('Install complete.')

In [ ]:
# === Cell 2: Kaggle Download ===
import os, json
os.makedirs('/root/.kaggle', exist_ok=True)

from google.colab import userdata
username = userdata.get('KAGGLE_USERNAME')
key = userdata.get('KAGGLE_KEY')
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': username, 'key': key}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print(f'Kaggle credentials set: {username}')

COMPETITION = 'super-ai-engineer-season-6-ocr-2569'
!kaggle competitions download -c {COMPETITION} -p /content/ 2>&1

# Unzip
import glob as glob_module
for zf in glob_module.glob('/content/*.zip'):
    print(f'Extracting: {zf}')
    !unzip -qo "{zf}" -d /content/
print('Done.')

In [ ]:
# === Cell 3: Imports & Configuration ===
import os, re, json, time, cv2
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from datetime import datetime
from tqdm.notebook import tqdm
from rapidfuzz import fuzz, process

# Auto-discover data paths
found = glob_module.glob('/content/**/images', recursive=True)
if found:
    IMAGE_DIR = Path(found[0])
    DATA_DIR = IMAGE_DIR.parent
else:
    DATA_DIR = Path('/content/data')
    IMAGE_DIR = DATA_DIR / 'images'

SAMPLE_LABELS_DIR = DATA_DIR / 'sample_labels'

# Find submission template
found_tmpl = glob_module.glob('/content/**/submission_template.csv', recursive=True)
SUBMISSION_TEMPLATE = Path(found_tmpl[0]) if found_tmpl else DATA_DIR / 'submission_template.csv'

OUTPUT_DIR = Path('/content/output')
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_FILE = OUTPUT_DIR / 'ocr_results.json'
PROGRESS_FILE = OUTPUT_DIR / 'progress.txt'

# Verify
images = sorted(IMAGE_DIR.glob('*.png'))
print(f'Images: {len(images)}')
print(f'Template: {SUBMISSION_TEMPLATE} (exists={SUBMISSION_TEMPLATE.exists()})')
print(f'Sample labels: {SAMPLE_LABELS_DIR.exists()}')

In [ ]:
# === Cell 4: Load Template & Group Documents ===
template_df = pd.read_csv(SUBMISSION_TEMPLATE)
print(f'Template: {template_df.shape}')
print(template_df.head())

# Group images by doc_id
def group_documents(image_dir):
    docs = defaultdict(list)
    pattern = re.compile(r'^(.+?)(?:_page(\d+))?\.png$')
    for p in sorted(image_dir.glob('*.png')):
        m = pattern.match(p.name)
        if m:
            doc_id = m.group(1)
            page = int(m.group(2)) if m.group(2) else 1
            docs[doc_id].append((page, str(p)))
    for d in docs:
        docs[d].sort()
    return dict(docs)

documents = group_documents(IMAGE_DIR)
print(f'Documents: {len(documents)}')

# Template lookup
template_doc_rows = template_df.groupby('doc_id').size().to_dict()
doc_expected = defaultdict(list)
for _, row in template_df.iterrows():
    doc_expected[row['doc_id']].append({
        'id': row['id'],
        'row_num': row['row_num'],
        'party_name': row['party_name'],
    })
for d in doc_expected:
    doc_expected[d].sort(key=lambda x: x['row_num'])

# Doc types
doc_types = {}
for d in documents:
    if 'party_list' in d.lower() or 'partylist' in d.lower():
        doc_types[d] = 'partylist'
    else:
        doc_types[d] = 'constituency'

c_count = sum(1 for v in doc_types.values() if v == 'constituency')
p_count = sum(1 for v in doc_types.values() if v == 'partylist')
print(f'Constituency: {c_count}, Party list: {p_count}')

In [ ]:
# === Cell 5: Initialize PaddleOCR ===
from paddleocr import PaddleOCR

# Thai OCR engine
ocr_engine = PaddleOCR(
    use_angle_cls=True,
    lang='th',
    show_log=False,
    use_gpu=True,
    det_db_thresh=0.3,
    det_db_box_thresh=0.5,
)
print('PaddleOCR initialized (Thai + GPU).')

In [ ]:
# === Cell 6: OCR Helper Functions ===

# Thai numeral conversion
THAI_TO_ARABIC = str.maketrans('\u0e50\u0e51\u0e52\u0e53\u0e54\u0e55\u0e56\u0e57\u0e58\u0e59', '0123456789')

def clean_number(text):
    """Extract clean number from OCR text."""
    s = str(text).translate(THAI_TO_ARABIC)
    s = re.sub(r'[^0-9]', '', s)
    return s if s else ''


def is_number_text(text):
    """Check if text contains a number (Thai or Arabic)."""
    cleaned = clean_number(text)
    return len(cleaned) > 0


def group_into_rows(boxes, y_threshold=15):
    """Group text boxes into rows by y-coordinate."""
    if not boxes:
        return []
    boxes_sorted = sorted(boxes, key=lambda b: b['cy'])
    rows = []
    current_row = [boxes_sorted[0]]
    for box in boxes_sorted[1:]:
        if abs(box['cy'] - current_row[0]['cy']) < y_threshold:
            current_row.append(box)
        else:
            rows.append(sorted(current_row, key=lambda b: b['cx']))
            current_row = [box]
    rows.append(sorted(current_row, key=lambda b: b['cx']))
    return rows


def ocr_page(img_path, engine):
    """OCR a single page, return list of text boxes with coordinates."""
    img = cv2.imread(img_path)
    if img is None:
        return []
    result = engine.ocr(img, cls=True)
    if result is None or result[0] is None:
        return []
    
    boxes = []
    for line in result[0]:
        bbox = line[0]  # [[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
        text = line[1][0]
        conf = line[1][1]
        cx = (bbox[0][0] + bbox[2][0]) / 2
        cy = (bbox[0][1] + bbox[2][1]) / 2
        x_min = min(p[0] for p in bbox)
        x_max = max(p[0] for p in bbox)
        y_min = min(p[1] for p in bbox)
        y_max = max(p[1] for p in bbox)
        boxes.append({
            'text': text,
            'conf': conf,
            'cx': cx, 'cy': cy,
            'x_min': x_min, 'x_max': x_max,
            'y_min': y_min, 'y_max': y_max,
            'width': x_max - x_min,
        })
    return boxes

print('Helper functions ready.')

In [ ]:
# === Cell 7: Table Extraction Logic ===

def find_table_region(boxes, img_height):
    """Find the table region based on content patterns.
    The table typically has rows with: number | text | text | number
    Returns filtered boxes that are likely part of the data table."""
    if not boxes:
        return []
    
    # Group into rows
    rows = group_into_rows(boxes, y_threshold=20)
    
    table_rows = []
    for row in rows:
        # A data row should have at least 2 elements
        if len(row) < 2:
            continue
        
        # Check if row has a number on the right side (vote count)
        rightmost = row[-1]
        num = clean_number(rightmost['text'])
        if num and len(num) >= 1:  # Has a vote-like number
            # Check if row also has Thai text (party name)
            has_thai = any(bool(re.search(r'[\u0e00-\u0e7f]', b['text'])) for b in row[:-1])
            if has_thai:
                table_rows.append(row)
    
    return table_rows


def extract_party_and_votes(table_rows, doc_type='constituency'):
    """Extract party names and vote counts from table rows."""
    results = []
    
    for row in table_rows:
        # The rightmost numeric text is the vote count
        # Work from right to left to find the vote
        vote_text = ''
        vote_idx = -1
        
        for i in range(len(row) - 1, -1, -1):
            num = clean_number(row[i]['text'])
            if num:
                vote_text = num
                vote_idx = i
                break
        
        if not vote_text:
            continue
        
        # Find party name: Thai text in the middle columns
        # For constituency: columns are number | name | party | votes
        # For partylist: columns are number | party | votes
        party_name = ''
        
        # Collect Thai text from boxes before the vote, excluding the leftmost number
        thai_texts = []
        for i, box in enumerate(row):
            if i >= vote_idx:
                break
            text = box['text'].strip()
            if re.search(r'[\u0e00-\u0e7f]', text):
                thai_texts.append((i, text, box['cx']))
        
        if thai_texts:
            if doc_type == 'constituency' and len(thai_texts) >= 2:
                # Party name is typically the last Thai text before votes
                party_name = thai_texts[-1][1]
            elif doc_type == 'partylist':
                # Party name is the main Thai text
                # Sometimes it's split into multiple boxes, take the longest or last
                party_name = thai_texts[-1][1]
            else:
                party_name = thai_texts[-1][1]
        
        # Also check if vote text might include the party number
        # (small numbers like 1-57 at the start)
        candidate_number = ''
        if row[0] != row[vote_idx]:
            first_num = clean_number(row[0]['text'])
            if first_num and 1 <= int(first_num) <= 60:
                candidate_number = first_num
        
        if party_name:
            results.append({
                'number': candidate_number,
                'party_name': party_name,
                'votes': vote_text,
            })
    
    return results


def extract_document_paddle(doc_id, page_paths, doc_type, engine):
    """Extract all party-vote pairs from a multi-page document."""
    all_results = []
    
    for page_num, img_path in page_paths:
        boxes = ocr_page(img_path, engine)
        if not boxes:
            continue
        
        # Get image height for region detection
        img = cv2.imread(img_path)
        img_height = img.shape[0] if img is not None else 3000
        
        # Find table rows
        table_rows = find_table_region(boxes, img_height)
        
        # Extract party names and votes
        page_results = extract_party_and_votes(table_rows, doc_type)
        all_results.extend(page_results)
    
    # Deduplicate (same party across pages)
    seen_parties = {}
    deduped = []
    for r in all_results:
        key = r['party_name']
        if key not in seen_parties:
            seen_parties[key] = r
            deduped.append(r)
    
    return deduped

print('Table extraction logic ready.')

In [ ]:
# === Cell 8: Test on sample document ===
test_doc = list(documents.keys())[0]
print(f'Testing on: {test_doc}')
print(f'Pages: {documents[test_doc]}')

test_results = extract_document_paddle(
    test_doc, documents[test_doc], doc_types[test_doc], ocr_engine
)

print(f'Extracted {len(test_results)} rows:')
for r in test_results:
    print(f"  {r['number']:>3s} | {r['party_name']:<30s} | {r['votes']}")

# Compare with sample label if available
label_file = SAMPLE_LABELS_DIR / f'{test_doc}.json'
if label_file.exists():
    with open(label_file, encoding='utf-8') as f:
        label = json.load(f)
    print(f'\nGround truth ({len(label["results"])} rows):')
    for r in label['results']:
        print(f"  {r.get('number', ''):>3} | {r.get('party', r.get('party_name', '')):30s} | {r['votes']}")

In [ ]:
# === Cell 9: Checkpoint & Save Functions ===

def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, encoding='utf-8') as f:
            return json.load(f)
    return {}

def save_checkpoint(results):
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)


def match_and_build_votes(results, template_df, doc_expected):
    """Match OCR results to template using fuzzy matching."""
    submission_votes = {}
    
    for doc_id, expected_rows in doc_expected.items():
        ocr_rows = results.get(doc_id, [])
        expected_parties = [e['party_name'] for e in expected_rows]
        
        if not ocr_rows:
            for exp in expected_rows:
                submission_votes[exp['id']] = '0'
            continue
        
        # Strategy 1: If row counts match, try positional matching
        if len(ocr_rows) == len(expected_rows):
            scores = []
            for ocr_row, exp in zip(ocr_rows, expected_rows):
                s = fuzz.ratio(ocr_row.get('party_name', ''), exp['party_name'])
                scores.append(s)
            avg_score = sum(scores) / len(scores) if scores else 0
            
            if avg_score > 50:
                for ocr_row, exp in zip(ocr_rows, expected_rows):
                    v = ocr_row.get('votes', '0')
                    submission_votes[exp['id']] = v if v else '0'
                continue
        
        # Strategy 2: Fuzzy name matching
        ocr_party_names = [r.get('party_name', '') for r in ocr_rows]
        matched_exp = set()
        
        for ocr_row in ocr_rows:
            ocr_name = ocr_row.get('party_name', '')
            if not ocr_name:
                continue
            
            best_match = process.extractOne(
                ocr_name,
                {i: e['party_name'] for i, e in enumerate(expected_rows) if i not in matched_exp},
                scorer=fuzz.ratio,
                score_cutoff=45
            )
            
            if best_match:
                idx = best_match[2]  # key = index
                v = ocr_row.get('votes', '0')
                submission_votes[expected_rows[idx]['id']] = v if v else '0'
                matched_exp.add(idx)
        
        # Fill unmatched with 0
        for i, exp in enumerate(expected_rows):
            if exp['id'] not in submission_votes:
                submission_votes[exp['id']] = '0'
    
    return submission_votes


def save_submission_csv(results, template_df, doc_expected, output_dir):
    """Generate and save submission CSV."""
    votes = match_and_build_votes(results, template_df, doc_expected)
    sub = template_df[['id']].copy()
    sub['votes'] = sub['id'].map(votes).fillna('0')
    csv_path = output_dir / 'submission.csv'
    sub.to_csv(csv_path, index=False)
    return csv_path


def update_progress(done, total, errors, start_time):
    """Update progress file."""
    elapsed = time.time() - start_time
    speed = done / elapsed if elapsed > 0 else 0
    eta = (total - done) / speed if speed > 0 else 0
    with open(PROGRESS_FILE, 'w') as f:
        f.write(f'OCR Progress\n')
        f.write(f'============\n')
        f.write(f'Completed: {done}/{total} documents ({done/total*100:.1f}%)\n')
        f.write(f'Errors: {len(errors)}\n')
        f.write(f'Speed: {speed:.2f} docs/sec\n')
        f.write(f'Elapsed: {elapsed/60:.1f} min\n')
        f.write(f'ETA: {eta/60:.1f} min\n')
        f.write(f'Last updated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')

print('Save functions ready.')

In [ ]:
# === Cell 10: Main Processing Loop ===
results = load_checkpoint()
print(f'Loaded {len(results)} existing results from checkpoint.')

doc_ids_sorted = sorted(documents.keys(), key=lambda d: (doc_types.get(d, 'z'), d))
todo = [d for d in doc_ids_sorted if d not in results]
print(f'Documents to process: {len(todo)} (skipped {len(results)} already done)')

SAVE_EVERY = 10
errors = []
start_time = time.time()

for i, doc_id in enumerate(tqdm(todo, desc='OCR')):
    doc_type = doc_types.get(doc_id, 'constituency')
    page_paths = documents[doc_id]
    
    try:
        rows = extract_document_paddle(doc_id, page_paths, doc_type, ocr_engine)
        results[doc_id] = rows
        
        expected = template_doc_rows.get(doc_id, -1)
        if expected > 0 and len(rows) != expected:
            print(f'  WARN {doc_id}: got {len(rows)}, expected {expected}')
    
    except Exception as e:
        errors.append(doc_id)
        print(f'  ERROR {doc_id}: {e}')
        results[doc_id] = []  # empty result, don't retry forever
    
    # Periodic save
    done = len(results)
    if done % SAVE_EVERY == 0 or i == len(todo) - 1:
        save_checkpoint(results)
        save_submission_csv(results, template_df, doc_expected, OUTPUT_DIR)
        update_progress(done, len(documents), errors, start_time)
        # Show inline progress
        elapsed = time.time() - start_time
        print(f'  [{done}/{len(documents)}] {elapsed/60:.1f}min elapsed, CSV saved')

# Final save
save_checkpoint(results)
save_submission_csv(results, template_df, doc_expected, OUTPUT_DIR)
update_progress(len(results), len(documents), errors, start_time)

elapsed = time.time() - start_time
print(f'\nDone! {len(results)}/{len(documents)} docs in {elapsed/60:.1f} min')
print(f'Errors: {len(errors)}')
if errors:
    print(f'Failed: {errors}')
print(f'Submission: {OUTPUT_DIR}/submission.csv')

In [ ]:
# === Cell 11: Validate Against Sample Labels ===

def levenshtein(s1, s2):
    if len(s1) < len(s2): return levenshtein(s2, s1)
    if len(s2) == 0: return len(s1)
    prev = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1):
        curr = [i + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(curr[j]+1, prev[j+1]+1, prev[j]+(c1!=c2)))
        prev = curr
    return prev[-1]

results = load_checkpoint()
votes_map = match_and_build_votes(results, template_df, doc_expected)

if SAMPLE_LABELS_DIR.exists():
    total_dist = 0
    total_pairs = 0
    
    for jf in sorted(SAMPLE_LABELS_DIR.glob('*.json')):
        with open(jf, encoding='utf-8') as f:
            label = json.load(f)
        doc_id = jf.stem
        label_results = label.get('results', [])
        
        print(f'\n=== {doc_id} ===')
        
        # Match label parties to template
        expected_rows = doc_expected.get(doc_id, [])
        
        for label_row in label_results:
            actual = str(label_row['votes'])
            party = label_row.get('party', label_row.get('party_name', ''))
            
            # Find this party's submission id
            predicted = '0'
            for exp in expected_rows:
                if fuzz.ratio(party, exp['party_name']) > 70:
                    predicted = votes_map.get(exp['id'], '0')
                    break
            
            dist = levenshtein(actual, predicted)
            total_dist += dist
            total_pairs += 1
            
            if dist > 0:
                print(f'  {party:30s}  actual={actual:>8s}  pred={predicted:>8s}  dist={dist}')
    
    if total_pairs > 0:
        print(f'\n=== Sample Validation ===')
        print(f'Pairs: {total_pairs}')
        print(f'Mean Levenshtein: {total_dist/total_pairs:.4f}')
else:
    print('No sample labels found.')

In [ ]:
# === Cell 12: Review Results ===
results = load_checkpoint()

# Stats
row_counts = {d: len(r) for d, r in results.items()}
expected_counts = {d: template_doc_rows.get(d, 0) for d in results}

match_count = sum(1 for d in results if row_counts[d] == expected_counts.get(d, -1))
mismatch = {d: (row_counts[d], expected_counts[d]) for d in results if row_counts[d] != expected_counts.get(d, -1)}

print(f'Total docs: {len(results)}')
print(f'Row count match: {match_count}/{len(results)}')
print(f'Row count mismatch: {len(mismatch)}')

if mismatch:
    print('\nMismatches (doc: got/expected):')
    for d, (got, exp) in sorted(mismatch.items())[:20]:
        print(f'  {d}: {got}/{exp}')

In [ ]:
# === Cell 13: Download Submission ===
from google.colab import files
files.download(str(OUTPUT_DIR / 'submission.csv'))

In [ ]:
# === Cell 14: Debug Specific Document (Optional) ===
# Uncomment and change doc_id to investigate specific documents

# DEBUG_DOC = 'constituency_10_1'
# page_paths = documents[DEBUG_DOC]
# 
# for page_num, img_path in page_paths:
#     print(f'\n--- Page {page_num}: {img_path} ---')
#     boxes = ocr_page(img_path, ocr_engine)
#     rows = group_into_rows(boxes, y_threshold=20)
#     for i, row in enumerate(rows):
#         texts = [f"{b['text']}({b['cx']:.0f})" for b in row]
#         print(f'  Row {i}: {" | ".join(texts)}')